# SleepWise AI — Standard Machine Learning Project

**Goal:** predict the dataset's *Quality of Sleep* score (1–10) using standard supervised machine learning regression.

We will clean the data, explore it, split it into training and test sets, train and compare several models, choose a model using 5-fold cross-validation MAE, evaluate it on held-out test data, and save the trained pipeline.

**Dataset:** [Kaggle — Sleep Health and Lifestyle Dataset](https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset) · [Public CSV mirror](https://github.com/broskunta/Sleep-Health-and-Lifestyle-Dataset/blob/main/Sleep_health_and_lifestyle_dataset.csv)

> Educational ML prototype only. Quality of Sleep is a self-reported dataset score, not a clinical measurement or diagnosis.

## 1. Upload the CSV and import libraries
Download the dataset CSV from the link above. Run this cell and upload the downloaded file.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Please upload the Sleep Health and Lifestyle CSV file.')
DATA_PATH = csv_files[0]
print('Using:', DATA_PATH)

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
TARGET = 'Quality of Sleep'
NUMERIC_FEATURES = ['Age', 'Sleep Duration', 'Physical Activity Level', 'Stress Level', 'Systolic BP', 'Diastolic BP', 'Heart Rate', 'Daily Steps']
CATEGORICAL_FEATURES = ['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']
OUTPUT_DIR = Path('sleepwise_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Inspect and clean the dataset
We drop the identifier, parse blood pressure into two numeric fields, treat blank sleep-disorder values as `None`, and remove duplicate records. The target column is kept separate from the input features.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()
print('Original shape:', df.shape)
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()}))

required = {'Age', 'Sleep Duration', TARGET, 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Blood Pressure', 'Heart Rate', 'Daily Steps', 'Gender', 'Occupation'}
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError('Wrong CSV or missing columns: ' + ', '.join(missing))

if 'Sleep Disorder' not in df.columns:
    df['Sleep Disorder'] = 'None'
else:
    df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None').astype(str).str.strip()
    df.loc[df['Sleep Disorder'].isin(['', 'nan', 'NaN']), 'Sleep Disorder'] = 'None'

bp = df['Blood Pressure'].astype(str).str.extract(r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)')
df['Systolic BP'] = pd.to_numeric(bp[0], errors='coerce')
df['Diastolic BP'] = pd.to_numeric(bp[1], errors='coerce')
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df.dropna(subset=[TARGET]).drop_duplicates().copy()
for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype('object')

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET].astype(float).copy()
print('Cleaned records:', len(df))
print('Input columns:', X.columns.tolist())
print('Target scores:', sorted(y.unique()))
display(y.describe().to_frame('Quality of Sleep'))

## 3. Exploratory data analysis
These plots help us understand the target distribution and numeric relationships. Correlation does not prove cause and effect.

In [ ]:
plt.figure(figsize=(7, 4))
y.value_counts().sort_index().plot(kind='bar')
plt.title('Distribution of Quality of Sleep')
plt.xlabel('Self-reported score')
plt.ylabel('Number of records')
plt.tight_layout()
plt.show()

corr = df[NUMERIC_FEATURES + [TARGET]].corr(numeric_only=True)
plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=75, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Numeric Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Train/test split and preprocessing
We keep 20% of records aside for final testing. Missing numeric values are median-imputed and scaled. Categorical values are imputed and one-hot encoded. Preprocessing is placed inside each model pipeline to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, NUMERIC_FEATURES),
    ('categorical', categorical_pipe, CATEGORICAL_FEATURES)
])

## 5. Train and compare standard regression models
We compare Linear Regression, Ridge, KNN, Decision Tree, Random Forest and Gradient Boosting. A mean baseline is included as a reference. The main selection metric is mean 5-fold cross-validation MAE on the training set; lower MAE is better.

In [ ]:
models = {
    'Mean Baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5, weights='distance'),
    'Decision Tree': DecisionTreeRegressor(max_depth=4, min_samples_leaf=3, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, loss='huber', random_state=RANDOM_STATE)
}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results, fitted_models = [], {}

for name, regressor in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', regressor)])
    cv_mae = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'CV MAE (mean)': cv_mae.mean(),
        'CV MAE (std)': cv_mae.std(),
        'Test MAE': mean_absolute_error(y_test, pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'Test R2': r2_score(y_test, pred)
    })
    fitted_models[name] = pipe
    print('Completed:', name)

comparison = pd.DataFrame(results).sort_values('CV MAE (mean)').reset_index(drop=True)
display(comparison.round(4))
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

## 6. Select the best model and report test performance
The best model is selected by lowest training cross-validation MAE. The held-out test set is used only for the final comparison. MAE and RMSE are error measures (lower is better); R² measures variance explained (higher is generally better).

In [ ]:
best_model_name = comparison.loc[0, 'Model']
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(X_test)
print('Selected model:', best_model_name)
print('Held-out Test MAE :', round(mean_absolute_error(y_test, best_predictions), 3))
print('Held-out Test RMSE:', round(np.sqrt(mean_squared_error(y_test, best_predictions)), 3))
print('Held-out Test R²  :', round(r2_score(y_test, best_predictions), 3))

predictions = pd.DataFrame({
    'Actual Quality of Sleep': y_test.to_numpy(),
    'Predicted Quality of Sleep': best_predictions,
    'Absolute Error': np.abs(y_test.to_numpy() - best_predictions)
})
display(predictions.head(10))
predictions.to_csv(OUTPUT_DIR / 'best_model_test_predictions.csv', index=False)

## 7. Visualize model comparison and explain feature importance
Permutation importance checks how the selected model's error changes when one input column is shuffled. It explains model reliance, not medical causation.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparison['Model'], comparison['CV MAE (mean)'])
plt.ylabel('Mean 5-fold CV MAE (lower is better)')
plt.title('SleepWise AI — Model Comparison')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=160)
plt.show()

perm = permutation_importance(best_model, X_test, y_test, scoring='neg_mean_absolute_error', n_repeats=10, random_state=RANDOM_STATE, n_jobs=1)
importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance (MAE increase)': perm.importances_mean,
    'Importance Std': perm.importances_std
}).sort_values('Importance (MAE increase)', ascending=False)
display(importance)
importance.to_csv(OUTPUT_DIR / 'permutation_feature_importance.csv', index=False)

top = importance.head(10).sort_values('Importance (MAE increase)')
plt.figure(figsize=(9, 5))
plt.barh(top['Feature'], top['Importance (MAE increase)'])
plt.xlabel('Increase in test MAE when feature is shuffled')
plt.title('Permutation Feature Importance')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=160)
plt.show()

## 8. Save the trained model and outputs
The saved object contains both preprocessing and the selected regression model. Use the same feature columns when making future predictions.

In [ ]:
joblib.dump(best_model, OUTPUT_DIR / 'sleepwise_best_model.joblib')
(OUTPUT_DIR / 'selected_model.txt').write_text(
    f'Selected model: {best_model_name}\nSelection basis: lowest mean 5-fold CV MAE on training data.\n',
    encoding='utf-8'
)

example_row = X_test.iloc[[0]]
print('Example actual score:', float(y_test.iloc[0]), '/10')
print('Example predicted score:', round(float(best_model.predict(example_row)[0]), 2), '/10')

print('Saved outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

import shutil
shutil.make_archive('sleepwise_outputs', 'zip', OUTPUT_DIR)
files.download('sleepwise_outputs.zip')
